# Document parsing with MinerU 2.5 and OpenVINO

[MinerU 2.5](https://github.com/opendatalab/MinerU) is a state-of-the-art document parsing engine maintained by [OpenDataLab](https://opendatalab.com/). Its 2.5 generation is built around a single 1.2 B parameter vision–language model — [`opendatalab/MinerU2.5-Pro-2604-1.2B`](https://huggingface.co/opendatalab/MinerU2.5-Pro-2604-1.2B) — that converts page images into structured Markdown / JSON in **two prompted steps**:

1. **Layout detection** — the whole page is sent to the model with the prompt `Layout Detection:`. The model emits special `<|box_start|>… <|box_end|><|ref_start|>type<|ref_end|>` tokens that describe every region's bounding box and type (text, title, table, formula, image, chart, …).
2. **Per-region content recognition** — every region is cropped and sent back with a region-specific prompt (`Text Recognition:`, `Table Recognition:`, `Formula Recognition:`, `Image Analysis:`).

On top of those two passes, a small post-processing layer fixes equation delimiters, normalises tables to HTML/OTSL, merges truncated paragraphs and finally renders Markdown via `json2md`.

The model itself is a Qwen2-VL architecture, so it can be exported to OpenVINO IR exactly like in the [`qwen2-vl`](../qwen2-vl/qwen2-vl.ipynb) notebook. In this tutorial we:

* convert and 4-bit weight-compress `MinerU2.5-Pro-2604-1.2B` with [Optimum Intel](https://github.com/huggingface/optimum-intel),
* run inference with [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai)'s `VLMPipeline`,
* drive the MinerU two-step pipeline through a small `OVMinerUClient` wrapper that reuses the official post-processing utilities,
* and expose the whole thing as a tiny Gradio document-to-Markdown demo.

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Select inference device](#Select-inference-device)
- [Build the OpenVINO MinerU client](#Build-the-OpenVINO-MinerU-client)
- [Run document parsing on a sample page](#Run-document-parsing-on-a-sample-page)
- [Parse a multi-page PDF](#Parse-a-multi-page-PDF)
- [Interactive Demo](#Interactive-Demo)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/mineru2.5/mineru2.5.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents)

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q -U "nncf>=2.15.0" "torch>=2.7" "torchvision>=0.22" "transformers>=4.53" "peft>=0.15.0" "Pillow" "pypdfium2" "gradio>=4.36,<6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q -U "openvino>=2026.0.0" "openvino-tokenizers>=2026.0.0" "openvino-genai>=2026.0.0"
%pip install -q --upgrade-strategy eager "optimum[openvino]>=1.26.0" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "mineru-vl-utils>=0.2.7"

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("mineru2.5.ipynb")

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents)

MinerU 2.5 ships as a PyTorch checkpoint with the `Qwen2VLForConditionalGeneration` architecture. We use the OpenVINO export integrated in 🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) to convert the model to OpenVINO IR.

The general command is:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

We additionally compress the weights to **INT4** with [NNCF](https://github.com/openvinotoolkit/nncf). Weight-only quantization preserves activations in floating point and is the recommended optimization for memory-bound LLMs/VLMs. See the [OpenVINO weight compression guide](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html) for more details. You can switch to `fp16` below if you prefer the reference precision.

<details>
    <summary><b>Click here for more details about weight compression</b></summary>

Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;
* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs.
</details>

In [ ]:
import ipywidgets as widgets

weight_format = widgets.Dropdown(
    options=["int4", "fp16"],
    value="int4",
    description="Weights:",
)
weight_format

In [ ]:
import json
import shutil
from huggingface_hub import snapshot_download
from cmd_helper import optimum_cli

pt_model_id = "opendatalab/MinerU2.5-Pro-2604-1.2B"
model_dir = Path(pt_model_id.split("/")[-1]) / weight_format.value.upper()

# Download a local snapshot so we can patch the config before exporting.
# The HF checkpoint sets ``tie_word_embeddings`` only inside ``text_config``;
# transformers v5 reads the top-level value (default ``False``) and therefore
# fails to tie ``lm_head.weight`` to ``embed_tokens.weight``, producing a
# randomly initialised LM head and gibberish output. Adding the flag at the
# top level fixes the export.
snap_dir = Path(snapshot_download(pt_model_id))
patched_dir = Path("./hf_snapshot") / pt_model_id.split("/")[-1]
if not patched_dir.exists():
    shutil.copytree(snap_dir, patched_dir)
    cfg_path = patched_dir / "config.json"
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if not cfg.get("tie_word_embeddings"):
        cfg["tie_word_embeddings"] = True
        cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")

if not model_dir.exists():
    optimum_cli(str(patched_dir), model_dir, additional_args={"task": "image-text-to-text", "weight-format": weight_format.value})

# Patch the OpenVINO detokenizer so MinerU layout tokens (<|box_start|>, etc.)
# survive decoding (they are marked special=True in the HF tokenizer config and
# would otherwise be stripped from the generated text).
from ov_mineru_helper import patch_detokenizer_for_mineru

patch_detokenizer_for_mineru(model_dir, patched_dir)

print(f"OpenVINO model is ready at: {model_dir.resolve()}")

## Select inference device
[back to top ⬆️](#Table-of-contents)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
device

## Build the OpenVINO MinerU client
[back to top ⬆️](#Table-of-contents)

`OVMinerUClient` wraps an `openvino_genai.VLMPipeline` and reproduces the official MinerU two-step pipeline:
* `prepare_for_layout` resizes the page to 1036×1036 and asks the VLM to emit the layout description;
* `parse_layout_output` regex-decodes the bounding boxes and region types;
* `prepare_for_extract` crops every region with the proper rotation / table-image masking and assembles the per-region prompts and sampling parameters;
* finally `post_process` from `mineru_vl_utils` cleans up equations, tables and paragraphs.

The whole post-processing layer is reused from the official `mineru-vl-utils` package — only the model call is swapped for OpenVINO.

In [ ]:
if not Path("ov_mineru_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/mineru2.5/ov_mineru_helper.py")
    open("ov_mineru_helper.py", "w").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/mineru2.5/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

from ov_mineru_helper import OVMinerUClient

client = OVMinerUClient(model_dir=model_dir, device=device.value)

## Run document parsing on a sample page
[back to top ⬆️](#Table-of-contents)

We download a single document image from the official MinerU repository, run the two-step pipeline, and display both the detected layout and the resulting Markdown.

In [ ]:
from IPython.display import Markdown, display

sample_url = "https://raw.githubusercontent.com/opendatalab/MinerU/master/demo/pdfs/demo1.pdf"
sample_pdf = Path("sample.pdf")
if not sample_pdf.exists():
    sample_pdf.write_bytes(requests.get(sample_url, timeout=60).content)

from ov_mineru_helper import pdf_to_images, render_blocks_overlay

pages = pdf_to_images(sample_pdf, dpi=200)
page = pages[0]
print(f"Sample page size: {page.size}")
page

In [ ]:
markdown, blocks = client.image_to_markdown(page)

print(f"Detected {len(blocks)} layout blocks. Block types:")
for b in blocks:
    print(f"  - {b['type']:<14}  bbox={[round(c, 3) for c in b['bbox']]}")

In [ ]:
render_blocks_overlay(page, blocks)

In [ ]:
display(Markdown(markdown))

## Parse a multi-page PDF
[back to top ⬆️](#Table-of-contents)

`OVMinerUClient.pdf_to_markdown` rasterises every page with [pypdfium2](https://github.com/pypdfium2-team/pypdfium2) and runs the two-step pipeline on each one. Pages are joined with horizontal rules in the resulting Markdown.

> **Note on images and tables:**
> - **Tables** are always recognised (OTSL → HTML conversion) and appear in the Markdown output.
> - **Images/charts** are skipped by default (`image_analysis=False`) — this matches the official MinerU 2.5 pipeline behaviour. To enable image/chart descriptions, pass `image_analysis=True` to `image_to_markdown()` or `pdf_to_markdown()` (see the Interactive Demo below for a toggle).

In [ ]:
def _progress(i, total):
    print(f"[{i}/{total}] processed")


full_md, _ = client.pdf_to_markdown(sample_pdf, dpi=200, progress_callback=_progress)
print(f"Total Markdown length: {len(full_md)} characters")

In [ ]:
display(Markdown(full_md[:8000] + ("…" if len(full_md) > 8000 else "")))

## Interactive Demo
[back to top ⬆️](#Table-of-contents)

Drop a PDF or an image of a document page into the Gradio UI below and inspect the parsed Markdown side by side.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(client)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/